# NCF with request features: early and late fusion

Two feature-aware neural candidate generators:

- **Early fusion** runs separate user and item towers over concatenated ID
  embeddings and request/article features, then joins the tower outputs.
- **Late fusion** merges the user and article ID embeddings first, then appends
  the request/article features before the final head.

Protocol: request-level BPR or BCE over the full `labels_train` pool (one clicked
positive and up to four exposed negatives redrawn per group each epoch). User features are pooled
`text64` and `entity100` history vectors from `request_profiles_<split>`.
Article features are the category one-hot and article `text64`. Popularity is
not a model feature.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
CG_DIR = ROOT / "5_Models" / "5_01_Candidate_Generation"
sys.path.insert(0, str(CG_DIR))
sys.path.insert(0, str(ROOT / "src"))

from cf_models import load_context, resolve_device
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    EpochNegativeSampler,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)
from newsrec.common import Settings

PROCESSED = ROOT / "1_Data_Preparation" / "processed"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = ROOT / "3_Representations" / "artifacts"
ARTIFACTS = ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
RESULTS = ROOT / "results"

TARGET_SPLIT = "validation"
FIT_SPLIT = "train"

TRIALS = ("late", "early")
M_VALUES = (50, 100, 200, 500, 750, 1000, 1500)
CANDIDATE_CAP = max(M_VALUES)

EMBED_DIM = 128
TEXT_DIM = 64
ENTITY_DIM = 100
HISTORY_DIM = TEXT_DIM + ENTITY_DIM
HOUR_NS = 3_600 * 10**9
POPULARITY_SPLITS = ["train", "validation"]
BRANCH_DIMS = (512, 256)
EARLY_HEAD_DIMS = (256, 128, 64)
LATE_ID_DIMS = (256, 128, 64)
LATE_HEAD_DIMS = (256, 128, 64, 32)
DROPOUT = 0.2
NUM_EPOCHS = 50
BATCH_SIZE = 32_768
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-6
SCORE_PAIRS = 131_072
SEED = 4222
RUN_LOSSES = ("bpr", "bce")
LOSS = RUN_LOSSES[0]
PATIENCE = 7
TUNE_QUANTILE = 0.9
TUNE_CANDIDATES = PROCESSED / "candidates_train.parquet"
DEVICE = resolve_device("auto")
COMPARISON_PATH = RESULTS / f"candidate_generation_comparison_{TARGET_SPLIT}.csv"

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULTS.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Target split:", TARGET_SPLIT)
print("Trials:", ", ".join(TRIALS))
print("Device:", DEVICE)
print("Torch:", torch.__version__)


## 2. Data preparation


In [ ]:
context = load_context(ROOT)
n_users, n_items = context.n_users, context.n_items

article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["article_row", "news_code", "category"],
).sort("article_row")

TEXT = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
ENTITY = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")

CATEGORY_VOCAB = sorted(article_index["category"].unique().to_list())
CATEGORY_DIM = len(CATEGORY_VOCAB)
category_code = {value: index for index, value in enumerate(CATEGORY_VOCAB)}
article_onehot = np.zeros((n_items, CATEGORY_DIM), dtype=np.float32)
article_onehot[
    np.arange(1, n_items),
    [category_code[value] for value in article_index["category"]],
] = 1.0

USER_EXTRA_DIM = HISTORY_DIM
ITEM_EXTRA_DIM = CATEGORY_DIM + TEXT_DIM


def load_profiles(split):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    return (
        pl.read_parquet(
            parts,
            columns=[
                "impression_uid",
                "user_code",
                "timestamp",
                "full_history_rows",
            ],
        )
        .sort("timestamp", maintain_order=True)
        .with_row_index("profile_row")
    )


def pool_profiles(history_rows, matrix, block=2_000):
    matrix = np.asarray(matrix)
    n, dim = len(history_rows), matrix.shape[1]
    result = np.zeros((n, dim), dtype=np.float32)
    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        lengths = np.fromiter(
            (len(rows) for rows in chunk), dtype=np.int64, count=len(chunk)
        )
        if lengths.sum() == 0:
            continue
        flat = np.concatenate(
            [np.asarray(rows, dtype=np.int64) for rows in chunk if len(rows)]
        )
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12
        if not valid.any():
            continue
        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)
        valid_groups = groups[valid]
        valid_vectors = vectors[valid]
        starts = np.r_[
            0, np.flatnonzero(valid_groups[1:] != valid_groups[:-1]) + 1
        ]
        pooled = np.add.reduceat(valid_vectors, starts, axis=0)
        pooled /= np.diff(np.r_[starts, len(valid_groups)])[:, None]
        norms = np.linalg.norm(pooled, axis=1)
        nonzero = norms > 0
        pooled[nonzero] /= norms[nonzero, None]
        result[start : start + len(chunk)][valid_groups[starts]] = pooled
    return result


def build_features(profiles):
    history_rows = profiles["full_history_rows"].to_list()
    features = np.zeros((len(history_rows), HISTORY_DIM), dtype=np.float32)
    features[:, :TEXT_DIM] = pool_profiles(history_rows, TEXT)
    features[:, TEXT_DIM:] = pool_profiles(history_rows, ENTITY)
    return features


train_profiles = load_profiles(FIT_SPLIT)
train_features = build_features(train_profiles)
target_profiles = load_profiles(TARGET_SPLIT)
target_features = build_features(target_profiles)

print(f"Users: {n_users:,}")
print(f"Articles: {n_items - 1:,}")
print(f"Categories: {CATEGORY_DIM}")
print(f"User extra dim: {USER_EXTRA_DIM}")
print(f"Item extra dim: {ITEM_EXTRA_DIM}")
print(f"Training requests: {train_profiles.height:,}")
print(f"Target requests: {target_profiles.height:,}")


In [ ]:
index_codes = article_index["news_code"].to_numpy().astype(np.int64)
row_of_code = np.zeros(int(index_codes.max()) + 1, dtype=np.int64)
row_of_code[index_codes] = article_index["article_row"].to_numpy().astype(
    np.int64
)

hourly = (
    pl.concat(
        [
            pl.scan_parquet(PROCESSED / f"candidates_{split}.parquet").select(
                "news_code",
                "clicked",
                (pl.col("timestamp").dt.epoch("ns") // HOUR_NS).alias("hour"),
            )
            for split in POPULARITY_SPLITS
        ]
    )
    .group_by("news_code", "hour")
    .agg(pl.col("clicked").cast(pl.Int64).sum())
    .collect()
)

MIN_HOUR = int(hourly["hour"].min())
MAX_HOUR = int(hourly["hour"].max())
N_HOURS = MAX_HOUR - MIN_HOUR + 1
CLICKS = np.zeros((N_HOURS, n_items), dtype=np.int32)
CLICKS[
    hourly["hour"].to_numpy().astype(np.int64) - MIN_HOUR,
    row_of_code[hourly["news_code"].to_numpy().astype(np.int64)],
] = hourly["clicked"].to_numpy().astype(np.int32)


def hour_index_of(timestamps_ns):
    return np.asarray(timestamps_ns, dtype=np.int64) // HOUR_NS - MIN_HOUR


def hourly_popularity(hour_index, item_rows):
    clk = CLICKS[hour_index - 1, item_rows].astype(np.float32)
    return np.log1p(clk)


ITEM_EXTRA = np.concatenate(
    [article_onehot, np.asarray(TEXT, dtype=np.float32)], axis=1
)
ITEM_EXTRA_TENSOR = torch.from_numpy(ITEM_EXTRA).to(DEVICE)

catalogue = pl.DataFrame(
    {
        "article_row": np.arange(1, n_items, dtype=np.int32),
        "news_code": context.news_code[1:].astype(np.int32),
        "available_from": context.available_from[1:],
    }
).sort(["available_from", "news_code"])
ARTICLE_ROWS_SORTED = catalogue["article_row"].to_numpy().astype(np.int64)
NEWS_SORTED = catalogue["news_code"].to_numpy().astype(np.int64)
AVAILABLE_SORTED = catalogue["available_from"].to_numpy().astype(
    "datetime64[ns]"
)
POS_IN_SORTED = np.full(n_items, -1, dtype=np.int64)
POS_IN_SORTED[ARTICLE_ROWS_SORTED] = np.arange(
    ARTICLE_ROWS_SORTED.size, dtype=np.int64
)

print("Item extra:", ITEM_EXTRA.shape)
print("Catalogue:", len(ARTICLE_ROWS_SORTED))
print(f"Rolling popularity hours: {N_HOURS:,}")


### Request-level BPR/BCE training data


In [ ]:
run_manifest = json.loads((FEATURES / "run_manifest.json").read_text())
if run_manifest.get("training_candidate_pool") != "complete":
    raise ValueError(
        "Per-epoch sampling needs the complete training pool. "
        "Rebuild stage 4 with the default training_negative_sampling='epoch'."
    )
CFG = Settings(**run_manifest["config"])

TRAINING_ROWS = (
    pl.read_parquet(sorted((FEATURES / "labels_train").glob("*.parquet")))
    .select("impression_uid", "news_code", "clicked")
    .to_pandas()
)
if TRAINING_ROWS["clicked"].isna().any() or not TRAINING_ROWS[
    "clicked"
].isin([0, 1]).all():
    raise ValueError("Training rows must be uniquely binary-labelled")

profile_lookup = train_profiles.select(
    "impression_uid", "timestamp", "profile_row", "user_code"
)
train_user_codes = train_profiles["user_code"].to_numpy().astype(np.int64)

positive_uids = np.unique(
    TRAINING_ROWS.loc[TRAINING_ROWS["clicked"].eq(1), "impression_uid"].to_numpy()
)
positive_profiles = profile_lookup.filter(
    pl.col("impression_uid").is_in(positive_uids)
)
cutoff_ns = chronological_cutoff(
    positive_profiles["timestamp"].to_numpy(), TUNE_QUANTILE
)
fit_profiles = chronological_mask(profile_lookup["timestamp"].to_numpy(), cutoff_ns)
fit_uids = set(
    profile_lookup.filter(pl.Series("fit", fit_profiles))["impression_uid"].to_list()
)
FIT_ROWS = TRAINING_ROWS[
    TRAINING_ROWS["impression_uid"].isin(fit_uids)
].reset_index(drop=True)
ALL_ROWS = TRAINING_ROWS
n_groups = int(TRAINING_ROWS["clicked"].sum())
EPOCH_SAMPLER = EpochNegativeSampler(ALL_ROWS, CFG)


def sample_epoch(epoch, *, fit_only):
    """Draw exposed negatives for one epoch and build feature-NCF arrays."""
    groups, _ = EPOCH_SAMPLER.draw(
        int(epoch), uids=fit_uids if fit_only else None
    )
    groups = (
        pl.from_pandas(groups)
        .join(profile_lookup, on="impression_uid", how="left")
        .with_row_index("group_row")
        .join(
            article_index.select(
                pl.col("news_code"),
                pl.col("article_row").alias("pos_row"),
            ),
            left_on="positive_news_code",
            right_on="news_code",
            how="left",
        )
    )
    if (
        groups["profile_row"].null_count()
        or groups["user_code"].null_count()
        or groups["pos_row"].null_count()
    ):
        raise ValueError("Sampled groups are missing training profiles or article rows")
    negative_long = (
        groups.select("group_row", "negative_news_codes")
        .explode("negative_news_codes")
        .rename({"negative_news_codes": "negative_news_code"})
        .drop_nulls("negative_news_code")
        .with_columns(pl.int_range(pl.len()).over("group_row").alias("neg_pos"))
        .join(
            article_index.select(
                pl.col("news_code").alias("negative_news_code"),
                pl.col("article_row").alias("neg_row"),
            ),
            on="negative_news_code",
            how="left",
        )
        .sort(["group_row", "neg_pos"])
    )
    if negative_long["neg_row"].null_count():
        raise ValueError("Sampled negatives are missing article rows")
    n_epoch_groups = groups.height
    NEG_MAX = int(groups["negative_news_codes"].list.len().max() or 0)
    if n_epoch_groups and NEG_MAX == 0:
        raise ValueError("Sampled training rows contain no exposed negatives")
    NEG_MATRIX = np.zeros((n_epoch_groups, NEG_MAX), dtype=np.int64)
    group_rows = negative_long["group_row"].to_numpy().astype(np.int64)
    NEG_MATRIX[
        group_rows, negative_long["neg_pos"].to_numpy().astype(np.int64)
    ] = negative_long["neg_row"].to_numpy().astype(np.int64)
    NEG_LEN = np.bincount(group_rows, minlength=n_epoch_groups).astype(np.int64)
    return (
        groups["user_code"].to_numpy().astype(np.int64),
        groups["profile_row"].to_numpy().astype(np.int64),
        groups["pos_row"].to_numpy().astype(np.int64),
        NEG_MATRIX,
        NEG_LEN,
    )


TUNE_NDCG_K = max_clicks_per_impression(TUNE_CANDIDATES)

tune = tuning_slates(
    TUNE_CANDIDATES,
    cutoff_ns,
    article_index,
    train_profiles.select("impression_uid", "profile_row"),
    max_impressions=None,
    seed=SEED + 1,
)

TUNE_PROFILE_IDX = tune["profile_row"].to_numpy().astype(np.int64)
TUNE_USER_IDS = train_user_codes[TUNE_PROFILE_IDX]
TUNE_ITEM_ROWS = tune["article_row"].to_numpy().astype(np.int64)
TUNE_CLICKED = tune["clicked"].to_numpy().astype(np.float32)
TUNE_UIDS = tune["impression_uid"].to_numpy()

print(f"Training rows: {len(TRAINING_ROWS):,}")
print(f"Training positives: {n_groups:,}")
print(f"Chronological fit rows: {len(FIT_ROWS):,} / {len(ALL_ROWS):,}")
print(f"Sampler negatives per positive: up to {CFG.negative_count}")
print(f"Tuning cutoff: {np.datetime64(cutoff_ns, 'ns')}")
print(f"Tuning rows: {tune.height:,}")
print(f"Tuning NDCG k: {TUNE_NDCG_K}")


## 3. Fusion models


In [ ]:
def mlp(dims, dropout=DROPOUT, final_activation=True):
    layers = []
    for index, (in_dim, out_dim) in enumerate(zip(dims[:-1], dims[1:])):
        layers.append(nn.Linear(in_dim, out_dim))
        if final_activation or index < len(dims) - 2:
            layers.extend([nn.GELU(), nn.Dropout(dropout)])
    return nn.Sequential(*layers)


class EarlyFusionNCF(nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        embed_dim,
        user_extra_dim,
        item_extra_dim,
        branch_dims,
        head_dims,
        dropout,
    ):
        super().__init__()
        self.branch_dim = branch_dims[-1]
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.item_embedding = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()
        self.input_dropout = nn.Dropout(dropout)
        self.fusion_dropout = nn.Dropout(dropout)
        self.user_tower = mlp(
            [embed_dim + user_extra_dim, *branch_dims], dropout
        )
        self.item_tower = mlp(
            [embed_dim + item_extra_dim, *branch_dims], dropout
        )
        self.head = mlp(
            [2 * self.branch_dim, *head_dims, 1], dropout, final_activation=False
        )

    def encode_users(self, user_ids, user_extra):
        return self.user_tower(
            self.input_dropout(
                torch.cat([self.user_embedding(user_ids), user_extra], dim=-1)
            )
        )

    def encode_items(self, item_ids, item_extra):
        return self.item_tower(
            self.input_dropout(
                torch.cat([self.item_embedding(item_ids), item_extra], dim=-1)
            )
        )

    def score_branches(self, user_branch, item_branch):
        return self.head(
            self.fusion_dropout(torch.cat([user_branch, item_branch], dim=-1))
        ).squeeze(-1)

    def score_pairs(self, user_ids, item_ids, user_extra, item_extra):
        return self.score_branches(
            self.encode_users(user_ids, user_extra),
            self.encode_items(item_ids, item_extra),
        )


class LateFusionNCF(nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        embed_dim,
        user_extra_dim,
        item_extra_dim,
        id_dims,
        head_dims,
        dropout,
    ):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.item_embedding = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()
        self.input_dropout = nn.Dropout(dropout)
        self.fusion_dropout = nn.Dropout(dropout)
        self.id_fusion = mlp([2 * embed_dim, *id_dims], dropout)
        self.head = mlp(
            [id_dims[-1] + user_extra_dim + item_extra_dim, *head_dims, 1],
            dropout,
            final_activation=False,
        )

    def fuse_embeddings(self, user_emb, item_emb):
        return self.id_fusion(
            self.input_dropout(torch.cat([user_emb, item_emb], dim=-1))
        )

    def score_fused(self, fused, user_extra, item_extra):
        return self.head(
            self.fusion_dropout(
                torch.cat([fused, user_extra, item_extra], dim=-1)
            )
        ).squeeze(-1)

    def score_pairs(self, user_ids, item_ids, user_extra, item_extra):
        return self.score_fused(
            self.fuse_embeddings(
                self.user_embedding(user_ids),
                self.item_embedding(item_ids),
            ),
            user_extra,
            item_extra,
        )


def build_model(trial):
    if trial == "early":
        return EarlyFusionNCF(
            n_users,
            n_items,
            EMBED_DIM,
            USER_EXTRA_DIM,
            ITEM_EXTRA_DIM,
            BRANCH_DIMS,
            EARLY_HEAD_DIMS,
            DROPOUT,
        ).to(DEVICE)
    return LateFusionNCF(
        n_users,
        n_items,
        EMBED_DIM,
        USER_EXTRA_DIM,
        ITEM_EXTRA_DIM,
        LATE_ID_DIMS,
        LATE_HEAD_DIMS,
        DROPOUT,
    ).to(DEVICE)


## 4. Training


In [ ]:
def train_epoch(
    model,
    optimizer,
    user_ids,
    profile_idx,
    pos_rows,
    neg_matrix,
    neg_len,
    user_features,
    item_extra_tensor,
    rng,
):
    model.train()
    order = rng.permutation(len(user_ids))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        users = torch.as_tensor(
            user_ids[batch], dtype=torch.long, device=DEVICE
        )
        user_extra = torch.as_tensor(
            user_features[profile_idx[batch]],
            dtype=torch.float32,
            device=DEVICE,
        )
        positives = torch.as_tensor(
            pos_rows[batch], dtype=torch.long, device=DEVICE
        )
        negatives = torch.as_tensor(
            neg_matrix[batch], dtype=torch.long, device=DEVICE
        )
        neg_lens = torch.as_tensor(
            neg_len[batch], dtype=torch.long, device=DEVICE
        )
        neg_mask = (
            torch.arange(negatives.shape[1], device=DEVICE)[None, :]
            < neg_lens[:, None]
        )
        users_expanded = users[:, None].expand(-1, negatives.shape[1])
        user_extra_expanded = user_extra[:, None, :].expand(
            -1, negatives.shape[1], -1
        ).reshape(-1, user_extra.shape[1])
        negatives_flat = negatives.reshape(-1)
        optimizer.zero_grad(set_to_none=True)
        positive_scores = model.score_pairs(
            users,
            positives,
            user_extra,
            item_extra_tensor[positives],
        )
        negative_scores = model.score_pairs(
            users_expanded.reshape(-1),
            negatives_flat,
            user_extra_expanded,
            item_extra_tensor[negatives_flat],
        ).reshape(len(batch), negatives.shape[1])
        if LOSS == "bpr":
            pair_loss = F.logsigmoid(
                positive_scores[:, None] - negative_scores
            )
            loss = -(pair_loss * neg_mask).sum() / neg_mask.sum()
        else:
            scores = torch.cat(
                [positive_scores, negative_scores[neg_mask]]
            )
            labels = torch.cat(
                [
                    torch.ones_like(positive_scores),
                    torch.zeros_like(negative_scores[neg_mask]),
                ]
            )
            loss = F.binary_cross_entropy_with_logits(scores, labels)
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)


def tune_ndcg(model):
    model.eval()
    scores = np.empty(len(TUNE_CLICKED), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(TUNE_CLICKED), SCORE_PAIRS):
            end = min(start + SCORE_PAIRS, len(TUNE_CLICKED))
            users = torch.as_tensor(
                TUNE_USER_IDS[start:end], dtype=torch.long, device=DEVICE
            )
            items = torch.as_tensor(
                TUNE_ITEM_ROWS[start:end], dtype=torch.long, device=DEVICE
            )
            user_extra = torch.as_tensor(
                train_features[TUNE_PROFILE_IDX[start:end]],
                dtype=torch.float32,
                device=DEVICE,
            )
            scores[start:end] = model.score_pairs(
                users,
                items,
                user_extra,
                ITEM_EXTRA_TENSOR[items],
            ).cpu().numpy()
    _, ndcg = mean_slate_metrics(
        TUNE_UIDS, TUNE_CLICKED, scores, TUNE_NDCG_K
    )
    return ndcg


def train_model(model, trial):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 1)
    training_log = []
    stopper = EarlyStopper(patience=PATIENCE)
    for epoch in range(1, NUM_EPOCHS + 1):
        started = time.time()
        user_ids, profile_idx, pos_rows, neg_matrix, neg_len = sample_epoch(
            epoch - 1, fit_only=True
        )
        loss = train_epoch(
            model,
            optimizer,
            user_ids,
            profile_idx,
            pos_rows,
            neg_matrix,
            neg_len,
            train_features,
            ITEM_EXTRA_TENSOR,
            rng,
        )
        ndcg = tune_ndcg(model)
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "tune",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": ndcg,
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(
            f"[{trial}] epoch {epoch:02d} loss={loss:.6f} "
            f"tune_ndcg@{TUNE_NDCG_K}={ndcg:.5f} ({elapsed:.1f}s)"
        )
        if stopper.step(ndcg, model, epoch):
            print(f"[{trial}] Early stopping")
            break

    stopper.restore(model)
    print(
        f"[{trial}] Best epoch: {stopper.best_epoch} "
        f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
    )

    final_model = build_model(trial)
    optimizer = torch.optim.AdamW(
        final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 2)
    for epoch in range(1, stopper.best_epoch + 1):
        started = time.time()
        user_ids, profile_idx, pos_rows, neg_matrix, neg_len = sample_epoch(
            epoch - 1, fit_only=False
        )
        loss = train_epoch(
            final_model,
            optimizer,
            user_ids,
            profile_idx,
            pos_rows,
            neg_matrix,
            neg_len,
            train_features,
            ITEM_EXTRA_TENSOR,
            rng,
        )
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "refit",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": float("nan"),
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(
            f"[{trial}] refit epoch {epoch:02d} "
            f"loss={loss:.6f} ({elapsed:.1f}s)"
        )
    return final_model, training_log, stopper


def save_model_artifact(
    trial, model, training_log, stopper, artifact_dir, model_name
):
    torch.save(
        {
            "state_dict": model.state_dict(),
            "category_vocab": CATEGORY_VOCAB,
            "config": {
                "trial": trial,
                "model": model_name,
                "n_users": n_users,
                "n_items": n_items,
                "embed_dim": EMBED_DIM,
                "user_extra_dim": USER_EXTRA_DIM,
                "item_extra_dim": ITEM_EXTRA_DIM,
                "category_dim": CATEGORY_DIM,
                "history_dim": HISTORY_DIM,
                "branch_dims": list(BRANCH_DIMS),
                "early_head_dims": list(EARLY_HEAD_DIMS),
                "late_id_dims": list(LATE_ID_DIMS),
                "late_head_dims": list(LATE_HEAD_DIMS),
                "dropout": DROPOUT,
                "max_epochs": NUM_EPOCHS,
                "best_epoch": int(stopper.best_epoch),
                "tune_ndcg": float(stopper.best_score),
                "ndcg_k": int(TUNE_NDCG_K),
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "seed": SEED,
                "objective": LOSS,
                "negative_sampling": "per-epoch exposed negatives",
                "negative_count": int(CFG.negative_count),
                "sampler_epoch_base": 0,
                "training_data": "labels_train full training pool",
            },
        },
        artifact_dir / "model.pt",
    )
    pl.DataFrame(training_log).write_csv(artifact_dir / "training_log.csv")
    print(f"[{trial}] Saved model:", artifact_dir / "model.pt")


TRAINED_TRIALS = {}
for loss in RUN_LOSSES:
    LOSS = loss
    for trial in TRIALS:
        model_name = f"ncf_features_{trial}_{loss}"
        artifact_dir = ARTIFACTS / model_name
        artifact_dir.mkdir(parents=True, exist_ok=True)
        model = build_model(trial)
        print(f"[{loss}/{trial}] Training {model_name}")
        model, training_log, stopper = train_model(model, trial)
        save_model_artifact(
            trial, model, training_log, stopper, artifact_dir, model_name
        )
        TRAINED_TRIALS[(loss, trial)] = {
            "loss": loss,
            "trial": trial,
            "model": model,
            "model_name": model_name,
            "artifact_dir": artifact_dir,
            "training_log": training_log,
            "stopper": stopper,
        }


## 5. Candidate generation and evaluation


In [ ]:
def popularity_select(order, excluded=None, k=CANDIDATE_CAP):
    excluded = set() if excluded is None else set(map(int, excluded))
    selected = [int(p) for p in order if int(p) not in excluded][:k]
    return np.asarray(selected, dtype=np.int64)


def popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]


def select_top_k(
    scores, popularity, news_code, k, excluded=None, fallback_order=None
):
    scores = np.asarray(scores, dtype=np.float32)
    n = scores.size
    if fallback_order is not None:
        return popularity_select(fallback_order, excluded=excluded, k=k)

    available = np.ones(n, dtype=bool)
    if excluded is not None and len(excluded):
        positions = np.asarray(excluded, dtype=np.int64)
        available[positions] = False
        scores = scores.copy()
        scores[positions] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return popularity_fill(available, popularity, news_code, k)

    take = min(k, n_finite)
    if take < n_finite:
        frontier = np.argpartition(-scores, take - 1)[:take]
        threshold = scores[frontier].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            ties = ties[
                np.lexsort((news_code[ties], -popularity[ties]))[:need]
            ]
        chosen_positions = np.r_[above, ties]
    else:
        chosen_positions = above

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity[chosen_positions],
            -scores[chosen_positions],
        )
    )
    selected = chosen_positions[order].astype(np.int64)

    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        selected = np.r_[
            selected,
            popularity_fill(
                remaining, popularity, news_code, k - selected.size
            ),
        ]

    return selected


In [ ]:
def encode_early_items(model, item_extra_tensor, batch=65_536):
    out = np.empty((n_items, model.branch_dim), dtype=np.float32)
    ids = torch.arange(n_items, dtype=torch.long, device=DEVICE)
    model.eval()
    with torch.no_grad():
        for start in range(0, n_items, batch):
            end = min(start + batch, n_items)
            out[start:end] = model.encode_items(
                ids[start:end], item_extra_tensor[start:end]
            ).cpu().numpy()
    return torch.from_numpy(out).to(DEVICE)


def score_early(model, user_branch, item_branch, max_pairs=SCORE_PAIRS):
    n_users = user_branch.shape[0]
    n_items_local = item_branch.shape[0]
    scores = np.empty((n_users, n_items_local), dtype=np.float32)
    items_per_chunk = min(n_items_local, max_pairs)
    users_per_chunk = max(1, max_pairs // items_per_chunk)
    with torch.no_grad():
        for u0 in range(0, n_users, users_per_chunk):
            u1 = min(u0 + users_per_chunk, n_users)
            for i0 in range(0, n_items_local, items_per_chunk):
                i1 = min(i0 + items_per_chunk, n_items_local)
                rows = u1 - u0
                cols = i1 - i0
                user_expand = user_branch[u0:u1, None, :].expand(
                    rows, cols, model.branch_dim
                )
                item_expand = item_branch[None, i0:i1, :].expand(
                    rows, cols, model.branch_dim
                )
                scores[u0:u1, i0:i1] = (
                    model.score_branches(user_expand, item_expand)
                    .cpu()
                    .numpy()
                )
    return scores


def score_late(
    model,
    user_ids,
    user_extra,
    item_emb,
    item_extra,
    max_pairs=SCORE_PAIRS,
):
    n_users = len(user_ids)
    n_items_local = item_emb.shape[0]
    scores = np.empty((n_users, n_items_local), dtype=np.float32)
    user_emb = model.user_embedding(
        torch.as_tensor(user_ids, dtype=torch.long, device=DEVICE)
    )
    items_per_chunk = min(n_items_local, max_pairs)
    users_per_chunk = max(1, max_pairs // items_per_chunk)
    with torch.no_grad():
        for u0 in range(0, n_users, users_per_chunk):
            u1 = min(u0 + users_per_chunk, n_users)
            for i0 in range(0, n_items_local, items_per_chunk):
                i1 = min(i0 + items_per_chunk, n_items_local)
                rows = u1 - u0
                cols = i1 - i0
                fused = model.fuse_embeddings(
                    user_emb[u0:u1, None, :].expand(rows, cols, EMBED_DIM),
                    item_emb[None, i0:i1, :].expand(rows, cols, EMBED_DIM),
                )
                scores[u0:u1, i0:i1] = model.score_fused(
                    fused,
                    user_extra[u0:u1, None, :].expand(
                        rows, cols, USER_EXTRA_DIM
                    ),
                    item_extra[None, i0:i1, :].expand(
                        rows, cols, ITEM_EXTRA_DIM
                    ),
                ).cpu().numpy()
    return scores


def score_target_split(trial, model, artifact_dir):
    recommendations_path = (
        artifact_dir
        / f"recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet"
    )
    timestamps = target_profiles["timestamp"].to_numpy().astype(
        "datetime64[ns]"
    )
    profile_hours = hour_index_of(timestamps.astype(np.int64))
    history_rows = target_profiles["full_history_rows"].to_list()
    has_history = np.fromiter(
        (len(rows) > 0 for rows in history_rows),
        dtype=bool,
        count=len(history_rows),
    )
    impression_uids = target_profiles["impression_uid"].to_numpy()
    target_user_codes = target_profiles["user_code"].to_numpy().astype(
        np.int64
    )

    starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
    ends = np.r_[starts[1:], len(timestamps)]
    batches = list(zip(starts, ends))
    print(f"[{trial}] Scoring requests: {len(timestamps):,}")

    model.eval()
    sorted_ids = torch.as_tensor(
        ARTICLE_ROWS_SORTED, dtype=torch.long, device=DEVICE
    )
    item_branch_sorted = item_emb_sorted = item_extra_sorted = None
    if trial == "early":
        item_branch_sorted = encode_early_items(
            model, ITEM_EXTRA_TENSOR
        ).index_select(0, sorted_ids)
    else:
        with torch.no_grad():
            item_emb_sorted = model.item_embedding(sorted_ids)
            item_extra_sorted = ITEM_EXTRA_TENSOR[sorted_ids]

    cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
    writer = CandidateWriter(recommendations_path)
    started = time.time()
    try:
        for batch_no, (start, end) in enumerate(batches, 1):
            timestamp = timestamps[start]
            batch_hour = int(profile_hours[start])
            n_eligible = int(
                np.searchsorted(AVAILABLE_SORTED, timestamp, side="right")
            )
            item_rows_batch = ARTICLE_ROWS_SORTED[:n_eligible]
            news_batch = NEWS_SORTED[:n_eligible]
            popularity_batch = hourly_popularity(
                batch_hour, item_rows_batch
            )
            fallback_order = np.lexsort((news_batch, -popularity_batch))

            user_ids = target_user_codes[start:end]
            user_extra = torch.as_tensor(
                target_features[start:end],
                dtype=torch.float32,
                device=DEVICE,
            )
            positions_tensor = torch.arange(
                n_eligible, dtype=torch.long, device=DEVICE
            )
            if trial == "early":
                with torch.no_grad():
                    user_branch = model.encode_users(
                        torch.as_tensor(
                            user_ids, dtype=torch.long, device=DEVICE
                        ),
                        user_extra,
                    )
                scores = score_early(
                    model,
                    user_branch,
                    item_branch_sorted[positions_tensor],
                )
            else:
                scores = score_late(
                    model,
                    user_ids,
                    user_extra,
                    item_emb_sorted[positions_tensor],
                    item_extra_sorted[positions_tensor],
                )

            selected_batch = np.empty(
                (end - start, CANDIDATE_CAP), dtype=np.int64
            )
            for local, profile_row in enumerate(range(start, end)):
                rows = history_rows[profile_row]
                if rows:
                    excluded = POS_IN_SORTED[
                        np.asarray(rows, dtype=np.int64)
                    ]
                    excluded = excluded[
                        (excluded >= 0) & (excluded < n_eligible)
                    ]
                else:
                    excluded = np.empty(0, dtype=np.int64)
                selected_batch[local] = select_top_k(
                    scores[local],
                    popularity_batch,
                    news_batch,
                    CANDIDATE_CAP,
                    excluded=excluded,
                    fallback_order=(
                        None
                        if has_history[profile_row]
                        else fallback_order
                    ),
                )

            writer.add(
                np.repeat(impression_uids[start:end], CANDIDATE_CAP),
                np.tile(cutoff_values, end - start),
                news_batch[selected_batch.reshape(-1)],
            )
            if batch_no % 1000 == 0 or batch_no == len(batches):
                print(
                    f"[{trial}] Scored {batch_no:,}/{len(batches):,} "
                    f"timestamp batches; {writer.rows:,} rows",
                    flush=True,
                )
    finally:
        writer.close()

    print(f"[{trial}] Wrote candidates in {time.time() - started:.1f}s")
    return recommendations_path


for loss in RUN_LOSSES:
    LOSS = loss
    for trial in TRIALS:
        state = TRAINED_TRIALS[(loss, trial)]
        state["recommendations_path"] = score_target_split(
            trial, state["model"], state["artifact_dir"]
        )


In [ ]:
from cg_evaluation import CandidateEvaluator

for loss in RUN_LOSSES:
    for trial in TRIALS:
        state = TRAINED_TRIALS[(loss, trial)]
        evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
        evaluation = evaluator.evaluate_to_csv(
            state["recommendations_path"],
            model_name=state["model_name"],
            m_values=M_VALUES,
            output_dir=RESULTS,
        )
        print(f"[{loss}/{trial}]")
        print(evaluation.sort("m"))


In [ ]:
comparison = pl.read_csv(COMPARISON_PATH)
model_names = [state["model_name"] for state in TRAINED_TRIALS.values()]
loss_table = (
    comparison.filter(pl.col("model").is_in(model_names))
    .filter(
        pl.col("metric").is_in(
            [
                "recall_pooled",
                "hit_rate",
                "catalogue_coverage",
                "personalisation",
                "long_tail_recall",
                "cold_article_recall",
            ]
        )
    )
    .pivot(on="model", index=["metric", "m"], values="value")
    .sort(["metric", "m"])
)
print(loss_table)


## Notes

- Request profiles are built from the causal `full_history_rows` history.
- BPR negatives are observed slate non-clicks, not uniform catalogue samples.
- Popularity is only used for tie-breaking and empty-history fallback.
- Early fusion precomputes the static item tower; late fusion evaluates the
  ID-fusion layers for every candidate pair.
